In [ ]:
# data cleaning

### Cleaning plan (notebook 03)
1. **Remove invalid rows.** Inspect the 34 rows with missing categorical values, then drop them. Check for fully duplicated rows.
2. **Define the target.** Convert `loan_status` into a binary default indicator. Remove loans without a known outcome (for example "Current") and record how each status is mapped.
3. **Apply the column selection.** Keep the candidate features, the target and `issue_d` (needed only for time-based splitting, not as a feature).
4. **Fix data types.** Convert `term` to a number, `emp_length` to an ordered numeric value with a separate missing category, the credit-line dates to dates, and `zip_code` to a text field.
5. **Assess missingness in numeric candidates.** Separate structural from random missingness, for example columns introduced only for later loans, and decide between a missing-value indicator, imputation or removal for each group.
6. **Handle joint applications.** Create an application-type indicator and combine the joint and individual income and DTI values.
7. **Check value validity.** Look for impossible or implausible values (for example in `dti`, `annual_inc`, `revol_util`, and FICO low above high) and decide on correction or capping after the EDA.
8. **Clean categorical fields.** Standardise text values and decide how to treat rare categories and high-cardinality fields (`addr_state`, `zip_code`).
9. **Save the result.** Write the cleaned dataset to `data/interim/` as parquet and log the row and column counts after each step.

### Rule for the next stage
Split the data (time-based, using `issue_d`) **before** fitting any imputer, encoder or scaler. These steps must be learned on the training data only, to avoid leakage.

In [ ]:
from pathlib import Path

import pandas as pd 
import numpy as np

pd.set_option("display.max_columns", None)

In [ ]:
DATA_PATH = Path("..") / "data" / "interim" / "accepted_2007_to_201804.parquet"
df_original = pd.read_parquet(DATA_PATH)

In [ ]:
df_original.shape

## creating a copy of csv file before data cleaning

In [ ]:
df = df_original.copy()

## Before the split (rules that don't learn anything from the data):

- Drop the 34 bad rows and duplicates.
- Define the target and remove loans with no known outcome.
- Drop the excluded columns.
- Fix data types (term, dates, emp_length mapping).
- Create the is-joint flag.
- Standardise text values.

In [ ]:
low_null_cols = [
    "term",
    "grade",
    "sub_grade",
    "home_ownership",
    "verification_status",
    "loan_status",
    "pymnt_plan",
    "purpose",
    "initial_list_status",
    "application_type",
    "hardship_flag",
    "disbursement_method",
    "debt_settlement_flag",
    "addr_state",
    "zip_code"
]

rows_with_missing = df[low_null_cols].isna().any(axis=1)

print(f"Rows affected: {rows_with_missing.sum()}")
print(f"Percentage affected: {rows_with_missing.mean() * 100:.4f}%")

In [ ]:
display(df.loc[rows_with_missing])

In [ ]:
missing_rows = df.loc[rows_with_missing]

null_percentage = missing_rows.isnull().mean().mean() * 100

print(f"Percentage of all values that are null: {null_percentage:.2f}%")

In [ ]:
# so there percentage of data in that rows are 97.27% null so there for removing them is appropriate

In [ ]:
rows_to_delete = df.index[rows_with_missing]
df.drop(index=rows_to_delete, inplace=True)
del rows_to_delete # removing the temporary variable

In [ ]:
df.shape # checking the shape after deletion 

In [ ]:
df['loan_status'].value_counts(dropna= False)

loan_status
Fully Paid                                             1076750
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

| `loan_status`                                           | Meaning                                                                 | Default?                           |
| ------------------------------------------------------- | ----------------------------------------------------------------------- | ---------------------------------- |
| **Fully Paid**                                          | Borrower completely repaid the loan                                     |  No                               |
| **Current**                                             | Loan is currently being paid on schedule                                |  Not known yet                   |
| **Charged Off**                                         | Lending Club considers the loan unlikely to be recovered                |  Yes                              |
| **Late (31-120 days)**                                  | Payment is 31–120 days overdue                                          |  High risk / potentially default |
| **In Grace Period**                                     | Payment is temporarily overdue but within grace period                  |  Not necessarily                 |
| **Late (16-30 days)**                                   | Payment is 16–30 days overdue                                           |  Not necessarily                 |
| **Does not meet the credit policy. Status:Fully Paid**  | Older loans that didn't meet current credit policy but were fully paid  |  No                               |
| **Does not meet the credit policy. Status:Charged Off** | Older loans that didn't meet current credit policy and were charged off |  Yes                              |
| **Default**                                             | Loan has formally defaulted                                             |  Yes                              |